# PX Patient Features (offset cohorts, NBA 3.1)

Generated from the updated patient feature skill only. Features: First Occurrence (pivot -> cross-join), Recency, Frequency A-D, Hospitalization A-D (`_PX` suffix).

* Cohorts, cutoffs and month bucketing come from `00_cohort_config` (`OFFSET_DAY = 7`, Skill Sec 2 / 10.8).
* Market basket: `PX_MARKET_BASKET` (Skill Sec 1.5). Not defined yet -> all events.
* `TEST_HCP_SAMPLE` widget: deterministic HCP sample (Skill Sec 12.2). `0` = full universe.
* Outputs: `notebook_output/patient_features_offset{OFFSET_DAY}[_test]/px_*`

In [0]:
%run /Users/patricio.guledjian.ext@bayer.com/kerendia_nba_3_1_develop_clone/00_cohort_config

In [0]:
%run ./00_Patient_Feature_Helpers_offset

In [0]:
dbutils.widgets.text("TEST_HCP_SAMPLE", "10000")
TEST_HCP_SAMPLE = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))

SOURCE      = "px"
EVENT_TYPE  = "px"
# Skill Sec 2.1 table (all PX files hold history from 2021)
CLAIMS_PATH      = BASE_QUERY_OUTPUT + "px_input_claims_all_events_features_input"             # First Occ + Recency
FREQ_CLAIMS_PATH = BASE_QUERY_OUTPUT + "px_input_claims_12M_and_first_occ_features_input"      # Frequency
HOSP_PATH        = BASE_QUERY_OUTPUT + "px_hospitalization_claims_all_events_features_input"   # pre-filtered upstream
PATIENT_LABEL      = "NUM_OF_PATIENTS_THE_HCP_DID_PROCEDURE_{event}_IN_LAST_{n}_DAYS"
HOSP_EVENT         = "ALL_PROCEDURES"
HOSP_PATIENT_LABEL = "NUM_OF_HOSPITALIZED_PATIENTS_THE_HCP_DID_PROCEDURE_IN_LAST_{n}_DAYS"
HOSP_COL_SUFFIX    = "_PX"
MARKET_BASKET      = get_market_basket(SOURCE)   # Skill Sec 1.5 (None until the baskets are added)

OUT_ROOT   = output_root(TEST_HCP_SAMPLE)
STAGE_ROOT = OUT_ROOT + f"_stage/{SOURCE}/"
print(f"OFFSET_DAY={OFFSET_DAY} | TEST_HCP_SAMPLE={TEST_HCP_SAMPLE} | OUT_ROOT={OUT_ROOT}")

In [0]:
max_data_date = standardize(spark.read.format("delta").load(CLAIMS_PATH), EVENT_TYPE).agg(F.max("EVENT_DATE")).collect()[0][0]
cohorts    = build_cohort_table(max_data_date)
max_cutoff = cohorts.agg(F.max("LOOKBACK_END_DATE")).collect()[0][0]
print(f"Max data date {max_data_date} -> {cohorts.count()} cohorts, last cutoff {max_cutoff}")
display(cohorts.orderBy("COHORT"))

hcp_universe = load_hcp_universe(TEST_HCP_SAMPLE, STAGE_ROOT)
claims = load_claims(CLAIMS_PATH, EVENT_TYPE, hcp_universe, max_cutoff, basket=MARKET_BASKET)
claims, event_keys = add_event_keys(claims, STAGE_ROOT + "claims", basket=MARKET_BASKET)
print(f"HCPs: {hcp_universe.count()} | claims: {claims.count():,} | date range:",
      claims.agg(F.min("EVENT_DATE"), F.max("EVENT_DATE")).collect()[0])

In [0]:
first_occurrence_features(claims, cohorts, event_keys, OUT_ROOT + f"{SOURCE}_first_occurrence_features", STAGE_ROOT)

In [0]:
recency_features(claims, cohorts, event_keys, [EVENT_TYPE], OUT_ROOT + f"{SOURCE}_recency_features", STAGE_ROOT)

In [0]:
# Frequency reads its own input file (Skill Sec 2.1 table)
freq_claims = load_claims(FREQ_CLAIMS_PATH, EVENT_TYPE, hcp_universe, max_cutoff, basket=MARKET_BASKET)
freq_claims, freq_event_keys = add_event_keys(freq_claims, STAGE_ROOT + "freq_claims", basket=MARKET_BASKET)
print("Frequency claims date range:", freq_claims.agg(F.min("EVENT_DATE"), F.max("EVENT_DATE")).collect()[0])
frequency_features(freq_claims, cohorts, freq_event_keys, PATIENT_LABEL, OUT_ROOT, SOURCE, STAGE_ROOT)

In [0]:
# Basket-filtered claims for both total and hospitalized counts (Skill Sec 1.5 rule 5)
total_claims = load_claims(CLAIMS_PATH, EVENT_TYPE, hcp_universe, max_cutoff, event_literal=HOSP_EVENT, basket=MARKET_BASKET)
hosp_claims  = load_claims(HOSP_PATH,   EVENT_TYPE, hcp_universe, max_cutoff, event_literal=HOSP_EVENT, basket=MARKET_BASKET)
hospitalization_features(total_claims, hosp_claims, cohorts, hcp_universe, HOSP_EVENT, HOSP_PATIENT_LABEL,
                         OUT_ROOT, SOURCE, STAGE_ROOT, col_suffix=HOSP_COL_SUFFIX)

In [0]:
validate_outputs(OUT_ROOT, SOURCE, claims)
dbutils.fs.rm(STAGE_ROOT, True)